Project Drishti · Any data. Any domain. One grammar.

Copyright (c) 2026 Ashutosh Sinha <ajsinha@gmail.com>.
All rights reserved.

PROPRIETARY AND CONFIDENTIAL.

This file is the confidential and proprietary property of Ashutosh Sinha.
Unauthorised copying, use, modification, distribution or disclosure of this
file, via any medium, is strictly prohibited except with the express prior
written permission of the copyright holder.

See the LICENSE file in the root of this repository for the full terms.

# 06 · A live intraday stream, and resuming it

With `stream: true` a request keeps sending: VaR sends the scope's scenario vector again after each intraday batch is applied. Each batch carries a **sequence** (the date's intraday stamp). Keep the last one; after a break ask again with `resume=` it, on this node or another, and the stream continues after it: nothing missed, nothing twice.

So that the notebook ends, it waits at most `DRISHTI_EXAMPLE_STREAM_WAIT` seconds (30) and takes at most `DRISHTI_EXAMPLE_STREAM_BATCHES` batches (2). A real program reads without a time limit.

*Read only:* this notebook only reads. Flight refuses every call that would write (`DoPut`, `DoExchange`), and nothing here asks for one.

**Setup.** `flight_setup.py` (beside this notebook) finds the one-file client `clients/python/drishti_flight.py` and reads the address, the token and the CA from the environment: `DRISHTI_FLIGHT_URL`, `DRISHTI_TOKEN` (or `DRISHTI_TOKEN_FILE`) and `DRISHTI_FLIGHT_CA`. No token is ever written in a notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))     # run Jupyter from examples/clients/python, where flight_setup.py is
from flight_setup import connect, meta, FlightRefused, SCOPE, DATE, STREAM_WAIT

## The request

The same Rachana measure call as a snapshot, with the stream flag that `stream()` adds.

In [ ]:
import os

MAX_BATCHES = int(os.environ.get("DRISHTI_EXAMPLE_STREAM_BATCHES", "2"))
live = connect(timeout=STREAM_WAIT)          # the time limit ends the notebook's wait, not the service's stream
req = live.request("measure", f"measure('var.scenarios', '{SCOPE}')")
req

## Follow it

Each item is `(sequence, pyarrow.RecordBatch)`. The first batch is the vector as it stands; later ones arrive after each intraday batch. Here: the worst scenario of each.

In [ ]:
def follow(stream, limit):
    """Up to `limit` batches; the last sequence seen (None when nothing came in time)."""
    last = None
    try:
        for i, (seq, batch) in enumerate(stream, 1):
            worst = min(batch.column("PNL").to_pylist())
            print(f"sequence {seq}: {batch.num_rows} scenarios, worst P&L {worst:,.0f}")
            last = seq
            if i >= limit:
                break
    except FlightRefused as e:
        if e.code:                            # a refusal: DRS-56xx says why
            raise
        print(f"no new batch within {STREAM_WAIT:.0f} seconds")
    finally:
        stream.close()
    return last

last = follow(live.stream(req), MAX_BATCHES)
print("last sequence held:", last)

## A break, then resume

The program stopped (a restart, a network break). It asks again with the last sequence it holds. Batches after it are sent; if nothing new has happened yet, the stream waits for the next intraday batch. A resume point older than the node can send from is `DRS-5609`: read a snapshot and start again.

In [ ]:
live.close()
live = connect(timeout=STREAM_WAIT)
if last is not None:
    last = follow(live.stream(req, resume=last), MAX_BATCHES) or last
print("last sequence held:", last)
live.close()